# Aula 13C — Simulador Interativo: Routing, Orchestration e Utility

Este laboratório usa **evidência do TIL quando disponível** e cai para **DEMO** explicitamente sintético quando não houver medições versionadas.

> Pergunta: **qual arquitetura entrega valor suficiente com qualidade, custo, latência e risco aceitáveis?**


## 📘 Glossário Vivo — conceitos-chave da Aula 13C

Use o Glossário Vivo durante o laboratório. Os conceitos abaixo formam a linguagem necessária para interpretar as decisões do simulador:

**[Baseline](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#baseline) · [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Model Orchestration](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orquestração-de-modelos) · [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) · [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) · [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia) · [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) · [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off)**

### 🧭 Durante o laboratório...

| Quando você estiver pensando em... | Consulte |
|---|---|
| estabelecer uma referência simples para comparação | [Baseline](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#baseline) |
| escolher modelos diferentes conforme a tarefa | [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) |
| combinar modelos ou etapas | [Model Orchestration](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orquestração-de-modelos) |
| decidir quando aceitar ou escalar uma resposta | [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) |
| entender quantos casos chegam à camada premium | [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) |
| equilibrar qualidade, custo e latência | [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) |
| analisar vários componentes trabalhando juntos | [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia) |
| estimar o impacto econômico das chamadas | [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) |
| aceitar ganhos em uma dimensão e perdas em outra | [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off) |

> Não memorize os termos isoladamente. Use os links quando o comportamento do simulador levantar uma dúvida conceitual.


## 1. Evidência antes da decisão

O simulador procura `til-model-evidence.csv`. Campos mínimos: `system`, `quality` (0–1), `cost_per_1000` e `latency_ms`. O contrato está em `data/model-evidence/README.md`.

**EVIDENCE** = medições versionadas. **DEMO** = proxies didáticos. Métrica simulada nunca deve ser apresentada como benchmark real.


### Estado de execução e evidência

Este notebook segue o padrão **headless-first** do TIL: a execução completa (`Run All`) não depende de interação humana. Os widgets são uma camada opcional e o cenário interativo só é calculado quando o aluno clica em **Simular cenário**.

O notebook foi validado em execução local headless e no Kaggle.

A trilha **AUTHOR / EVIDENCE** já produziu duas camadas de evidência:

- `EDU-ORCH-001`: modelos isolados medidos em `til-model-evidence.csv`;
- `EDU-ORCH-002`: quatro cascades medidos em `til-routing-evidence.csv`.

As linhas de routing usam `evidence_status = measured-recovered`: os números vieram de execução real no Kaggle, mas o CSV original precisou ser reconstruído a partir do output observado após perda do artefato da sessão. Isso não equivale a `estimated` e também não deve ser apresentado como `measured` pleno.

A Aula 13C opera em **STUDENT MODE**: consome a evidência pronta para interpretação rápida. Se algum artefato não estiver disponível no ambiente, o notebook mantém fallback `DEMO` explicitamente sintético.

> Evidência de modelos isolados e evidência de routing são camadas diferentes. O notebook não transforma resultados cross-run em comparação estritamente controlada e não interpola novas medições a partir das cascades recuperadas.


### STUDENT MODE — evidência pronta

A Aula 13C não treina o Transformer. Ela consome evidências versionadas produzidas pelos experimentos `EDU-ORCH-001` e `EDU-ORCH-002`.

```text
experimento
→ evidência
→ aula
→ exploração rápida
```

📚 Glossário Vivo: [Evidência](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#evidência) · [Roteamento de modelos](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Quality gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Taxa de escalonamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) · [Função de utilidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade).


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

REQ={"system","quality","cost_per_1000","latency_ms"}
ROUTING_REQ=REQ|{"threshold","escalation_rate"}

DEMO=pd.DataFrame([
["tfidf_naive_bayes",.82,.20,12,"demo"],
["distilbert",.92,.75,58,"demo"],
["llm_or_human_review",.96,4.00,650,"demo"]],
columns=["system","quality","cost_per_1000","latency_ms","evidence_status"])

def candidates(filename):
    p=[Path(f"data/model-evidence/{filename}"),
       Path(f"../data/model-evidence/{filename}"),
       Path(f"../../data/model-evidence/{filename}")]
    k = Path("/kaggle/input")
    if k.exists():
        for dataset_dir in k.iterdir():
            if dataset_dir.is_dir():
                candidate = dataset_dir / filename
                if candidate.exists():
                    p.append(candidate)
    return p

def load_table(filename, required, numeric):
    for p in candidates(filename):
        if p.exists():
            d=pd.read_csv(p)
            if d.empty or not required.issubset(d.columns):
                continue
            for c in numeric:
                d[c]=pd.to_numeric(d[c],errors="coerce")
            d=d.dropna(subset=list(required))
            if "quality" in d:
                d=d[d.quality.between(0,1)]
            if "cost_per_1000" in d:
                d=d[d.cost_per_1000.ge(0)]
            if "latency_ms" in d:
                d=d[d.latency_ms.ge(0)]
            if len(d):
                return d.reset_index(drop=True),str(p)
    return None,None

EVIDENCE,MODEL_SOURCE=load_table(
    "til-model-evidence.csv",REQ,
    ["quality","cost_per_1000","latency_ms"]
)
if EVIDENCE is None or len(EVIDENCE)<2:
    EVIDENCE=DEMO.copy()
    MODEL_MODE="DEMO"
    MODEL_SOURCE="fallback sintético"
else:
    MODEL_MODE="EVIDENCE"

ROUTING,ROUTING_SOURCE=load_table(
    "til-routing-evidence.csv",ROUTING_REQ,
    ["threshold","quality","cost_per_1000","latency_ms","escalation_rate"]
)
ROUTING_MODE="EVIDENCE" if ROUTING is not None else "DEMO"
MODE="EVIDENCE" if MODEL_MODE=="EVIDENCE" else "DEMO"

display(Markdown(
    f"### Modelos: **{MODEL_MODE}**  \n`{MODEL_SOURCE}`  \n"
    f"### Routing: **{ROUTING_MODE}**  \n`{ROUTING_SOURCE or 'fallback sintético'}`"
))
display(EVIDENCE)
if ROUTING is not None:
    display(ROUTING[["system","threshold","quality","cost_per_1000","latency_ms","escalation_rate","evidence_status"]])


## 2. Sistema composto

As linhas de `til-model-evidence.csv` são candidatos `single`. Quando `til-routing-evidence.csv` está disponível, o `cascade` usa **uma das quatro configurações realmente medidas** pelo `EDU-ORCH-002` (`0.60`, `0.70`, `0.80`, `0.90`).

O controle de **quality gate** seleciona um threshold medido; a aula não interpola uma nova cascade entre pontos. O status `measured-recovered` permanece visível para preservar a proveniência.

Se o artefato de routing não estiver disponível, a aula cai para um fallback `DEMO` explicitamente sintético. Esse fallback serve apenas à mecânica pedagógica e nunca deve ser apresentado como benchmark.

📚 Glossário: [Quality Gate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#quality-gate) · [Escalation Rate](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-escalonamento) · [Model Routing](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#roteamento-de-modelos) · [Trade-off](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#trade-off).


In [ ]:
def mm(s):
    a,b=s.min(),s.max()
    return pd.Series(np.zeros(len(s)),index=s.index) if np.isclose(a,b) else (s-a)/(b-a)

def cascade(g):
    cheap=EVIDENCE.sort_values(["cost_per_1000","latency_ms"]).iloc[0]
    premium=EVIDENCE.sort_values(["quality","cost_per_1000"],ascending=[False,True]).iloc[0]

    if ROUTING is not None:
        idx=(ROUTING["threshold"]-float(g)).abs().idxmin()
        r=ROUTING.loc[idx]
        return {
          "system":r.system,
          "quality":float(r.quality),
          "cost_per_1000":float(r.cost_per_1000),
          "latency_ms":float(r.latency_ms),
          "escalation_rate":float(r.escalation_rate),
          "threshold":float(r.threshold),
          "evidence_status":r.get("evidence_status","measured-recovered"),
          "cheap":cheap.system,"premium":premium.system}

    rate=float(np.clip(.05+.90*g,0,1))
    return {
      "system":f"demo_cascade:{cheap.system}→{premium.system}",
      "quality":(1-rate)*cheap.quality+rate*premium.quality,
      "cost_per_1000":cheap.cost_per_1000+rate*premium.cost_per_1000,
      "latency_ms":cheap.latency_ms+rate*premium.latency_ms,
      "escalation_rate":rate,
      "threshold":float(g),
      "evidence_status":"demo",
      "cheap":cheap.system,"premium":premium.system}

def evaluate(wq,wc,wl,g):
    w=np.array([wq,wc,wl],float)
    w=np.ones(3) if np.isclose(w.sum(),0) else w
    w/=w.sum()

    d=EVIDENCE[["system","quality","cost_per_1000","latency_ms"]].copy()
    d["architecture"]="single"
    d["escalation_rate"]=0.
    d["threshold"]=np.nan
    d["evidence_status"]=EVIDENCE.get("evidence_status",MODEL_MODE.lower())

    x=cascade(g)
    row=pd.DataFrame([{
        "system":x["system"],
        "quality":x["quality"],
        "cost_per_1000":x["cost_per_1000"],
        "latency_ms":x["latency_ms"],
        "architecture":"cascade",
        "escalation_rate":x["escalation_rate"],
        "threshold":x["threshold"],
        "evidence_status":x["evidence_status"],
    }])
    d=pd.concat([d,row],ignore_index=True)
    d["utility"]=w[0]*mm(d.quality)-w[1]*mm(d.cost_per_1000)-w[2]*mm(d.latency_ms)
    return d.sort_values("utility",ascending=False).reset_index(drop=True),x,w


## 3. 🎛️ Simulador

Os controles mudam **prioridades**, não as medições. Ajuste qualidade, custo e latência. Quando a matriz de routing está disponível, o controle de gate escolhe entre os thresholds realmente medidos; nenhum novo ponto é interpolado.

📚 Glossário: [Utility Function](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#função-de-utilidade) · [Cost per Inference](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#custo-por-inferência) · [Compound AI System](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sistema-composto-de-ia).


In [ ]:
sty={"description_width":"140px"}
lay=widgets.Layout(width="95%")

q=widgets.IntSlider(value=60,min=0,max=100,step=5,description="Qualidade",style=sty,layout=lay,continuous_update=False)
c=widgets.IntSlider(value=25,min=0,max=100,step=5,description="Custo",style=sty,layout=lay,continuous_update=False)
l=widgets.IntSlider(value=15,min=0,max=100,step=5,description="Latência",style=sty,layout=lay,continuous_update=False)

if ROUTING is not None:
    thresholds=sorted(float(x) for x in ROUTING["threshold"].unique())
    default_threshold=min(thresholds,key=lambda x:abs(x-.80))
    g=widgets.SelectionSlider(
        options=[(f"{x:.2f}",x) for x in thresholds],
        value=default_threshold,
        description="Quality gate",
        style=sty,layout=lay,continuous_update=False
    )
else:
    g=widgets.FloatSlider(value=.45,min=0,max=1,step=.05,description="Quality gate",style=sty,layout=lay,continuous_update=False)

preset=widgets.ToggleButtons(options=[("Equilibrado","b"),("Qualidade","q"),("Custo","c"),("Latência","l")],value="b")
run_button=widgets.Button(description="Simular cenário",button_style="primary",icon="play")
out=widgets.Output()

def render(_=None):
    d,x,w=evaluate(q.value,c.value,l.value,g.value)
    with out:
        clear_output(wait=True)
        win=d.iloc[0]
        model_note="evidência versionada" if MODEL_MODE=="EVIDENCE" else "proxies sintéticos"
        routing_note=x["evidence_status"]
        display(Markdown(
            f"**Vencedor do cenário:** `{win.system}` · modelos: **{model_note}**  \n"
            f"**Cascade:** `{x['system']}` · threshold **{x['threshold']:.2f}** · "
            f"escalonamento **{x['escalation_rate']:.2%}** · status **{routing_note}**"
        ))
        z=d[["system","architecture","quality","cost_per_1000","latency_ms","escalation_rate","threshold","evidence_status","utility"]].copy()
        z.insert(0,"rank",range(1,len(z)+1))
        display(z.round(4))

        fig,ax=plt.subplots(figsize=(8,5))
        for _,r in d.iterrows():
            ax.scatter(r.cost_per_1000,r.quality,s=90)
            ax.annotate(r.system,(r.cost_per_1000,r.quality),xytext=(5,5),textcoords="offset points")
        ax.set(xlabel="Custo / 1.000 inferências",ylabel="Qualidade",title=f"Custo × qualidade — modelos {MODEL_MODE} / routing {ROUTING_MODE}")
        ax.grid(alpha=.25)
        plt.show()
        plt.close(fig)

        fig,ax=plt.subplots(figsize=(8,4))
        ax.bar(d.system,d.utility)
        ax.axhline(0,lw=1)
        ax.set_ylabel("Utility")
        plt.xticks(rotation=25,ha="right")
        plt.show()
        plt.close(fig)

def choose(ch):
    if ch.get("name")!="value":
        return
    vals={"b":(60,25,15),"q":(85,10,5),"c":(40,50,10),"l":(40,10,50)}
    q.value,c.value,l.value=vals[ch["new"]]

preset.observe(choose,names="value")
run_button.on_click(render)

display(widgets.VBox([
    widgets.HTML(f"<b>Modelos:</b> {MODEL_MODE} · <b>Routing:</b> {ROUTING_MODE}<br>Ajuste os parâmetros e clique em <b>Simular cenário</b>."),
    preset,q,c,l,g,run_button,out
]))


## 4. Sensibilidade do gate

Compare os thresholds medidos para observar **retorno decrescente**: mais escalonamento pode elevar custo e latência sem ganho proporcional de qualidade.

No artefato atual, `0.80` apresentou o maior F1 entre os quatro thresholds observados, mas isso é um resultado contextual — não um ótimo universal.

📚 Glossário: **Quality Gate**, **Escalation Rate**, **Trade-off**.


In [ ]:
rows=[]
gates=(sorted(float(x) for x in ROUTING["threshold"].unique())
       if ROUTING is not None else np.linspace(0,1,21))

for gate in gates:
    d,x,w=evaluate(60,25,15,gate)
    r=d[d.architecture.eq("cascade")].iloc[0]
    rows.append([x["threshold"],r.escalation_rate,r.quality,r.cost_per_1000,r.latency_ms,r.utility,x["evidence_status"]])

S=pd.DataFrame(rows,columns=["threshold","escalation_rate","quality","cost_per_1000","latency_ms","utility","evidence_status"])
display(S.round(4))

for y,label in [("quality","Qualidade"),("cost_per_1000","Custo / 1.000")]:
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot(S.threshold,S[y],marker="o")
    ax.set(xlabel="Quality gate / threshold",ylabel=label,title=f"Threshold × {label}")
    ax.grid(alpha=.25)
    plt.show()
    plt.close(fig)


## 5. Como a evidência chega à aula

O TIL separa produção e consumo de evidência:

`experimento autoral → treinamento/medição → artefato versionado → aula do aluno → interpretação/decisão`

Nesta aula:

- `EDU-ORCH-001` fornece `til-model-evidence.csv`;
- `EDU-ORCH-002` fornece `til-routing-evidence.csv`;
- a Aula 13C não repete o treinamento longo do Transformer;
- `measured` e `measured-recovered` permanecem distintos;
- resultados de execuções diferentes não são promovidos automaticamente a comparação estritamente controlada.

O contrato de proveniência está em `data/model-evidence/README.md`.


## 6. Desafio e síntese

Teste políticas de **qualidade primeiro**, **custo primeiro** e **latência primeiro**. Responda: existe melhor modelo universal ou **melhor sistema condicionado ao contexto**?

Sua conclusão deve citar pelo menos quatro conceitos do **Glossário Vivo** e separar **evidência medida**, **hipótese** e **proxy didático**.

`métrica → custo do erro → threshold/abstenção → quality gate → routing → orchestration → compound AI system → decisão baseada em evidências`


## 7. Evidência real ativada — EDU-ORCH-001 + EDU-ORCH-002

A Aula 13C consome duas camadas de evidência produzidas pelo próprio TIL.

O `EDU-ORCH-001 — Model evidence: baseline clássico vs Transformer` mediu, no mesmo conjunto de teste (`n = 3807`):

| Métrica | TF-IDF + NB | DistilBERT |
|---|---:|---:|
| F1 macro | 0.9099 | 0.9300 |
| Accuracy | 0.9241 | 0.9406 |
| Latência média | 1.09 ms | 34.38 ms |
| Proxy de runtime / 1000 | 0.030 s | 52.00 s |

O `EDU-ORCH-002 — Routing Evidence Matrix` mediu quatro cascades:

| Threshold | F1 macro | Accuracy | Escalation rate | Runtime proxy / 1000 | Latência média |
|---:|---:|---:|---:|---:|---:|
| 0.60 | 0.9178 | 0.9307 | 3.94% | 4.49 s | 5.13 ms |
| 0.70 | 0.9248 | 0.9364 | 8.25% | 8.97 s | 12.20 ms |
| 0.80 | 0.9330 | 0.9433 | 14.42% | 15.42 s | 22.28 ms |
| 0.90 | 0.9318 | 0.9422 | 23.46% | 25.15 s | 34.01 ms |

As linhas de routing estão marcadas como `measured-recovered`. O threshold `0.80` apresentou o maior F1 **entre esses quatro pontos medidos**, sem implicar ótimo universal.

### O que muda pedagogicamente?

A pergunta deixa de ser apenas:

> “qual modelo tem maior qualidade?”

e passa a ser:

> “qual arquitetura entrega utilidade suficiente para determinada combinação de qualidade, latência, custo e risco?”

Isso conecta diretamente:

```text
Model
→ Selection
→ Routing
→ Orchestration
→ Utility
→ Compound AI System
```

### Próximo marco experimental

O próximo passo é **ampliar a matriz de evidências**, preservando a separação entre AUTHOR / EVIDENCE e STUDENT.

Novos experimentos poderão acrescentar, de forma reproduzível:

```text
novos modelos
+ outros hardwares
+ outros datasets
+ novas tarefas
+ custo monetário quando observável
+ novos thresholds ou políticas de routing
+ LLMs
+ revisão humana
```

Cada nova linha deverá preservar o contrato de proveniência definido em `data/model-evidence/README.md`.


## 8. Case Study — de Model Intelligence para Agentic Systems

A evolução recente dos modelos de fronteira mostra que **capacidade do modelo** e **capacidade do sistema** não são a mesma coisa.

O GPT-6 Astra é um bom estudo de caso porque combina raciocínio com uso de ferramentas, navegação, computer use e execução de tarefas longas. Isso desloca a análise de uma pergunta centrada apenas no modelo:

```text
Qual modelo é mais inteligente?
```

para uma pergunta de engenharia de sistemas:

```text
Qual sistema consegue concluir a tarefa
com qualidade, custo, latência, supervisão e risco aceitáveis?
```

### 8.1 Seis conceitos que não devem ser confundidos

| Conceito | Pergunta |
|---|---|
| **Benchmark** | O que foi medido, em qual tarefa e sob quais condições? |
| **Capability** | O que o modelo consegue fazer? |
| **Autonomy** | Por quanto tempo e com quanta independência ele consegue agir? |
| **Utility** | O resultado compensa custo, latência, risco e supervisão? |
| **Risk** | O que pode dar errado quando o sistema deixa de apenas responder e passa a agir? |
| **Marketing** | O que a manchete ou comunicação comercial está simplificando? |

Um benchmark alto não implica, sozinho, autonomia ampla. E autonomia maior não implica, sozinha, utility maior.

### 8.2 O case Astra

A OpenAI reporta para o GPT-6 Astra resultados muito altos em benchmarks específicos, incluindo FrontierMath Tier 4, ARC-AGI-3 e ExploitBench. Esses números devem ser lidos como evidência sobre tarefas particulares, não como uma medida única de “inteligência humana geral”.

Mais importante para esta aula é a mudança arquitetural:

```text
Model
  ↓
Selection
  ↓
Routing
  ↓
Orchestration
  ↓
Tools
  ↓
Computer Use
  ↓
Agentic Execution
  ↓
Observability
  ↓
Human Oversight
  ↓
Utility
```

Nesse desenho, o modelo passa a fazer parte de um **sistema de execução**. O valor final depende não apenas da qualidade da resposta, mas também da capacidade de agir, observar o resultado, corrigir a trajetória e permanecer dentro do escopo autorizado.

### 8.3 Utility em sistemas agentes

Na Aula 13C já usamos utility para combinar qualidade, custo e latência. Em sistemas agentes, outras dimensões passam a importar.

Uma extensão didática possível é:

```text
utility =
  task_completion_value
+ quality
- cost
- latency
- execution_risk
- supervision_cost
```

Essa expressão não é uma fórmula universal. Ela serve para mostrar que a utility de um agente pode piorar mesmo quando o modelo subjacente fica mais capaz.

### 8.4 Risco muda quando o modelo passa a agir

A OpenAI classifica o GPT-6 Astra no nível **Critical** de capacidade de cibersegurança segundo seu Preparedness Framework.

O ponto pedagógico não é transformar esta aula em uma aula de cibersegurança. É perceber que:

```text
maior capability
+ ferramentas
+ autonomia
= nova superfície de risco
```

Em sistemas agentes, observabilidade e supervisão humana deixam de ser componentes periféricos e passam a fazer parte da própria arquitetura.

### 8.5 Como ler uma manchete tecnológica

Use a manchete como ponto de partida, não como conclusão.

Diante de uma afirmação como “melhor que um ser humano”, pergunte:

1. em qual benchmark ou tarefa?
2. qual população ou conjunto de teste?
3. qual harness, ferramenta ou acesso estava disponível?
4. o resultado foi replicado independentemente?
5. a métrica mede capacidade, autonomia ou utility?
6. quais riscos aparecem quando a capacidade é colocada em execução?

Essa decomposição é mais útil para engenharia do que aceitar ou rejeitar a manchete em bloco.

### Fontes do case

- OpenAI — *GPT-6 Astra: A new generation of intelligence*.
- OpenAI — *Safety overview: GPT-6 Astra*.
- OpenAI — *Path to Astra: critical capabilities and frontier safeguards*.
- Matéria da IGN fornecida como provocação jornalística para o estudo de caso.

> Regra do TIL: fontes primárias sustentam os fatos técnicos; a manchete jornalística é usada como objeto de análise crítica.


### Exercício — Benchmark, Capability, Autonomy, Utility, Risk ou Marketing?

Classifique cada afirmação abaixo na categoria predominante e justifique em uma frase.

| Afirmação | Categoria esperada para discussão |
|---|---|
| “99,9% no ARC-AGI-3” | Benchmark |
| “Consegue operar interfaces e ferramentas” | Capability |
| “Executa uma sequência longa com pouca intervenção” | Autonomy |
| “Vale a pena usar o sistema para esta tarefa?” | Utility |
| “Pode realizar uma ação não autorizada” | Risk |
| “Melhor que um ser humano” | Marketing / afirmação que exige decomposição |

#### Desafio

Considere dois sistemas:

```text
Sistema A
- qualidade alta
- custo baixo
- sem ferramentas
- nenhuma ação externa

Sistema B
- qualidade ligeiramente maior
- custo muito maior
- computer use
- autonomia longa
- exige monitoramento e confirmação humana
```

Não existe um vencedor universal.

Explique em que contexto o Sistema A teria maior utility e em que contexto o Sistema B poderia justificar o custo e a superfície adicional de risco.

#### Pergunta de fechamento

Quando um modelo passa a usar ferramentas e executar ações, **qual nova métrica você adicionaria ao simulador da Aula 13C?**

Exemplos possíveis:

- task completion rate;
- intervention rate;
- rollback rate;
- unauthorized-action rate;
- supervision time;
- recovery success rate;
- end-to-end task cost.

A resposta deve justificar por que a nova métrica muda uma decisão de routing ou orchestration.



---

## Continue no TIL

← **[Anterior: Aula 13B — Metric Scenario Lab](https://www.kaggle.com/code/pedrogentil/til-13-metric-scenario-lab)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 14 — LLM Foundations](https://www.kaggle.com/code/pedrogentil/til-14-llm-foundations)** →
